# Day 14: KV Cache — Prefix Caching & Cache-Aware Routing
> *Inference Engineering* — Chapter 5.3 | Philip Kiely, Baseten Books 2026

**Layer:** Runtime | **Prerequisite:** Day 07 (vLLM/PagedAttention)


## Concept Overview

Prefix caching stores previously computed KV states for common prompt prefixes, allowing reuse across requests without recomputation. A system prompt shared across 1000 concurrent users only needs to be computed once. Cache-aware routing directs new requests to the GPU instance that already holds the relevant KV cache blocks, maximizing cache hit rates. Together, these techniques convert O(prompt_len) compute per request into O(prompt_len / cache_hit_rate) amortized cost.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import hashlib
from collections import OrderedDict

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Hash-Based Prefix Cache

Cache keys are computed from token sequence hashes. Block-level hashing (per 16 tokens) enables partial prefix matches.


In [ ]:
class PrefixCache:
    def __init__(self, max_blocks=1000, block_size=16):
        self.cache = OrderedDict()  # hash -> kv_block (LRU)
        self.max_blocks = max_blocks
        self.block_size = block_size
        self.hits = 0
        self.misses = 0

    def _block_hash(self, tokens):
        return hashlib.sha256(bytes(tokens)).hexdigest()[:16]

    def get(self, token_sequence):
        """Return (num_cached_tokens, list_of_block_ids)."""
        blocks = []
        for start in range(0, len(token_sequence), self.block_size):
            block_tokens = token_sequence[start:start+self.block_size]
            if len(block_tokens) < self.block_size:
                break  # only cache complete blocks
            h = self._block_hash(block_tokens)
            if h in self.cache:
                blocks.append(self.cache[h])
                self.cache.move_to_end(h)  # LRU update
                self.hits += 1
            else:
                self.misses += 1
                break  # stop at first miss (prefix must be contiguous)
        return len(blocks) * self.block_size, blocks

    def put(self, token_sequence, block_data_fn):
        """Insert blocks for completed sequence."""
        for start in range(0, len(token_sequence), self.block_size):
            block_tokens = token_sequence[start:start+self.block_size]
            if len(block_tokens) < self.block_size:
                break
            h = self._block_hash(block_tokens)
            if h not in self.cache:
                if len(self.cache) >= self.max_blocks:
                    self.cache.popitem(last=False)  # evict LRU
                self.cache[h] = f'kv_block_{h}'

# Simulate requests with a shared system prompt
cache = PrefixCache(max_blocks=500)
system_prompt = list(range(128))  # 128 token system prompt
np.random.seed(42)

print('Prefix cache simulation with shared system prompt (128 tokens):')
saved_tokens = 0
for i in range(100):
    # 80% of requests share the system prompt; 20% are unique
    if np.random.random() < 0.8:
        tokens = system_prompt + list(np.random.randint(128, 1000, size=np.random.randint(10, 50)))
    else:
        tokens = list(np.random.randint(0, 1000, size=np.random.randint(10, 100)))
    cached_len, _ = cache.get(tokens)
    saved_tokens += cached_len
    cache.put(tokens, lambda t: f'kv_{t}')

total_tokens = sum(len(system_prompt) + 30 for _ in range(100))
print(f'Total requests: 100')
print(f'Cache hits: {cache.hits}, misses: {cache.misses}')
print(f'Tokens saved (not recomputed): {saved_tokens}')
print(f'Compute savings: {saved_tokens/total_tokens*100:.0f}%')


## 2. Cache-Aware Routing

When multiple GPU instances hold copies of the KV cache, routing new requests to the instance with the highest prefix overlap minimizes total compute.


In [ ]:
class CacheAwareRouter:
    def __init__(self, num_workers):
        self.workers = [PrefixCache(max_blocks=200) for _ in range(num_workers)]
        self.num_workers = num_workers
        self.load = [0] * num_workers

    def route(self, token_sequence, max_load=20):
        """Route to worker with highest cache coverage (if load allows)."""
        best_worker = 0
        best_cached = 0

        for i, worker in enumerate(self.workers):
            if self.load[i] >= max_load:
                continue
            cached_len, _ = worker.get(token_sequence)
            if cached_len > best_cached:
                best_cached = cached_len
                best_worker = i

        self.load[best_worker] += 1
        return best_worker, best_cached

    def complete(self, worker_id, token_sequence):
        self.load[worker_id] -= 1
        self.workers[worker_id].put(token_sequence, None)

# Simulate routing
np.random.seed(42)
router = CacheAwareRouter(num_workers=4)

# Seed each worker with different system prompts
system_prompts = {
    0: list(range(0, 128)),
    1: list(range(128, 256)),
    2: list(range(256, 384)),
    3: list(range(384, 512)),
}
for worker_id, sp in system_prompts.items():
    router.workers[worker_id].put(sp, None)

routing_decisions = []
for i in range(200):
    # Request uses one of the 4 system prompts
    sp_idx = np.random.randint(0, 4)
    tokens = system_prompts[sp_idx] + list(np.random.randint(512, 2000, size=30))
    worker, cached = router.route(tokens)
    routing_decisions.append({'correct_worker': sp_idx, 'routed_to': worker, 'cached_tokens': cached})
    router.complete(worker, tokens)

correct = sum(1 for d in routing_decisions if d['correct_worker'] == d['routed_to'])
mean_cached = np.mean([d['cached_tokens'] for d in routing_decisions])
print(f'Cache-aware routing results:')
print(f'  Routed to optimal worker: {correct}/200 ({correct/2:.0f}%)')
print(f'  Mean cached tokens per request: {mean_cached:.0f}')


## 3. KV Cache Memory Sizing

Calculating KV cache requirements for production deployments.


In [ ]:
def kv_cache_size_gb(num_layers, num_kv_heads, d_head, max_seq_len, dtype_bytes=2):
    """KV cache memory per sequence in GB."""
    return 2 * num_layers * num_kv_heads * d_head * max_seq_len * dtype_bytes / 1e9

models = [
    ('Llama-3-8B',  32, 8,  128, 'GQA'),
    ('Llama-3-70B', 80, 8,  128, 'GQA'),
    ('Mistral-7B',  32, 8,  128, 'GQA'),
    ('Qwen2.5-7B',  28, 4,  128, 'GQA'),
]
seq_lens = [4096, 8192, 32768, 131072]

print(f'{"Model":<20}', end='')
for sl in seq_lens:
    print(f' {sl//1024}K seq (GB)', end='')
print()
print('-' * 80)
for name, layers, kv_heads, d_head, attn_type in models:
    print(f'{name:<20}', end='')
    for sl in seq_lens:
        gb = kv_cache_size_gb(layers, kv_heads, d_head, sl)
        print(f' {gb:>12.3f}', end='')
    print()


## Experiments: Try These

1. **Hit rate vs system prompt length**: Vary system prompt length (64 to 4096 tokens) and measure prefix cache hit rate for 1000 requests.
2. **LRU vs LFU eviction**: Implement both LRU and LFU eviction policies. Which achieves higher hit rates on realistic workloads (Zipf prefix distribution)?
3. **Consistent hashing for routing**: Replace the greedy router with consistent hashing (prefix hash → worker). How does it handle worker failures?


## Key Takeaways

- Prefix caching stores completed KV blocks keyed by token hash; block-level hashing enables partial prefix matches and LRU eviction.
- With an 80% hit rate on a 128-token system prompt, prefix caching saves >60% of prefill compute across all requests.
- Cache-aware routing directs requests to the worker with the most matching KV blocks, maximizing hit rates under load.
- GQA (Grouped Query Attention) dramatically reduces KV cache size vs MHA — Llama-3-8B uses 8 KV heads vs 32 query heads, 4x smaller KV cache.

## References
- *Inference Engineering* Ch 5.3 — Philip Kiely, Baseten Books 2026
- vLLM prefix caching documentation
- SGLang RadixAttention paper
